In [1]:
# %%
# ============================================================
# BULLPEN PIPELINE (v3) — team-level relief quality
# Approach B: pull ALL pitches per team, define bullpen as
# "every pitcher except that game's starter", aggregate to
# team-game level, then roll AND expand over the team's games.
# Change SEASON, run top to bottom, repeat for the other year.
# ============================================================
import time
import pandas as pd
from pybaseball import statcast

# ---------------- CONFIG ----------------
SEASON = 2026   # <-- change to 2026 and re-run

SEASON_START_DATES = {2025: '2025-03-27', 2026: '2026-03-25'}
PULL_WINDOWS       = {2025: ('2025-03-27', '2025-10-01'),
                      2026: ('2026-03-25', '2026-07-01')}
RWIN = 10   # bullpen rolling window (last N team games)

DATA_DIR = '/Users/jackdiamond/Documents/Sports_Models/MLB/O/U/Data'
GAMES_FEATURES_PATH = f'{DATA_DIR}/games_{SEASON}_features_v2.2.csv'
BULLPEN_STARTS_OUT  = f'bullpen_team_games_{SEASON}.csv'
GAMES_V3_OUT        = f'{DATA_DIR}/games_{SEASON}_features_v3.csv'

SEASON_START = pd.Timestamp(SEASON_START_DATES[SEASON])
PULL_START, PULL_END = PULL_WINDOWS[SEASON]

# Statcast team abbreviations (what statcast(team=...) expects)
STATCAST_TEAMS = ['AZ','ATL','BAL','BOS','CHC','CWS','CIN','CLE','COL','DET',
                  'HOU','KC','LAA','LAD','MIA','MIL','MIN','NYM','NYY','ATH',
                  'PHI','PIT','SD','SF','SEA','STL','TB','TEX','TOR','WSH']

SWING = ['swinging_strike','swinging_strike_blocked','foul','foul_tip','hit_into_play']
WHIFF = ['swinging_strike','swinging_strike_blocked']

# %%
# ============================================================
# 1. PULL all pitches per team, isolate BULLPEN, aggregate to team-game
# ============================================================
def bullpen_team_games(pitch_df, team_abbr):
    """From one team's full pitch data, keep only pitches THAT TEAM'S pitchers
    threw (not opponents'), drop the game's starter, aggregate relievers to
    one row per game."""
    df = pitch_df.copy()

    # Which team was pitching on each pitch: home team pitches in the TOP, away in BOTTOM
    df['pitching_team'] = df['home_team'].where(df['inning_topbot'] == 'Top', df['away_team'])
    # Keep only pitches thrown BY this team
    df = df[df['pitching_team'] == team_abbr].copy()
    if df.empty:
        return pd.DataFrame()

    # Identify the STARTER per game = the pitcher who threw this team's very first pitch.
    # Sort by game, inning, and pitch order so the first row per game is the opener.
    df = df.sort_values(['game_pk', 'inning', 'at_bat_number', 'pitch_number'])
    starters = df.groupby('game_pk')['pitcher'].first().rename('starter_id')
    df = df.merge(starters, on='game_pk')

    # Bullpen = every pitch NOT thrown by that game's starter
    bp = df[df['pitcher'] != df['starter_id']].copy()
    if bp.empty:
        return pd.DataFrame()

    bp['is_swing'] = bp['description'].isin(SWING)
    bp['is_whiff'] = bp['description'].isin(WHIFF)
    bp['is_pa']    = bp['events'].notna()

    g = bp.groupby(['game_date','game_pk']).agg(
        bp_batters_faced=('is_pa','sum'),
        bp_strikeouts=('events', lambda x: (x=='strikeout').sum()),
        bp_walks=('events', lambda x: (x=='walk').sum()),
        bp_swings=('is_swing','sum'),
        bp_whiffs=('is_whiff','sum'),
        bp_pitches=('description','size'),
    ).reset_index()
    g['team'] = team_abbr
    g['bp_K_pct']     = g['bp_strikeouts'] / g['bp_batters_faced']
    g['bp_BB_pct']    = g['bp_walks'] / g['bp_batters_faced']
    g['bp_whiff_pct'] = g['bp_whiffs'] / g['bp_swings'].replace(0, pd.NA)
    return g

all_team_games, pull_failed = [], []
for i, team in enumerate(STATCAST_TEAMS, 1):
    try:
        pitches = statcast(PULL_START, PULL_END, team=team)
        if pitches.empty:
            pull_failed.append(team); continue
        all_team_games.append(bullpen_team_games(pitches, team))
    except Exception as e:
        pull_failed.append(team)
        print(f"  FAILED {team}: {e}")
    print(f"  ...{i}/{len(STATCAST_TEAMS)} teams pulled")
    time.sleep(0.5)

bullpen = pd.concat(all_team_games, ignore_index=True)
bullpen['game_date'] = pd.to_datetime(bullpen['game_date'])
bullpen = bullpen[bullpen['game_date'] >= SEASON_START].reset_index(drop=True)

bullpen.to_csv(BULLPEN_STARTS_OUT, index=False)
print(f"\nDone. {len(bullpen)} team-game bullpen lines from "
      f"{bullpen['team'].nunique()} teams. Failed: {pull_failed}")

# %%
# ============================================================
# 2. ROLLING + EXPANDING(talent) bullpen quality per team
#    shift(1) = leak-free (never sees current game's bullpen)
# ============================================================
bullpen = pd.read_csv(BULLPEN_STARTS_OUT, parse_dates=['game_date'])
bullpen = bullpen.sort_values(['team','game_date']).reset_index(drop=True)

for stat in ['bp_K_pct', 'bp_BB_pct', 'bp_whiff_pct']:
    # rolling (recent form)
    bullpen[f'{stat}_roll{RWIN}'] = (
        bullpen.groupby('team')[stat]
        .transform(lambda x: x.shift(1).rolling(RWIN, min_periods=3).mean())
    )
    # expanding (season talent)
    bullpen[f'{stat}_talent'] = (
        bullpen.groupby('team')[stat]
        .transform(lambda x: x.shift(1).expanding(min_periods=3).mean())
    )

print(bullpen[['team','game_date','bp_whiff_pct',
               f'bp_whiff_pct_roll{RWIN}','bp_whiff_pct_talent']].head(15))

# %%
# ============================================================
# 3. FIX abbreviations (Statcast -> Baseball-Reference) + JOIN to games
# ============================================================
statcast_to_bref = {'AZ':'ARI','CWS':'CHW','KC':'KCR','SD':'SDP',
                    'SF':'SFG','TB':'TBR','WSH':'WSN'}
bullpen['team'] = bullpen['team'].replace(statcast_to_bref)

games = pd.read_csv(GAMES_FEATURES_PATH)
games['game_date'] = pd.to_datetime(games['Date'])

bp_cols = [f'bp_K_pct_roll{RWIN}', f'bp_BB_pct_roll{RWIN}', f'bp_whiff_pct_roll{RWIN}',
           'bp_K_pct_talent', 'bp_BB_pct_talent', 'bp_whiff_pct_talent']
bp_slim = bullpen[['game_date','team'] + bp_cols].copy()

# Join HOME bullpen
home_bp = bp_slim.rename(columns={c: f'home_{c}' for c in bp_cols})
games = games.merge(
    home_bp.assign(_t=home_bp['team']).drop(columns='team'),
    left_on=['game_date','home_team'], right_on=['game_date','_t'], how='left'
).drop(columns='_t')

# Join AWAY bullpen
away_bp = bp_slim.rename(columns={c: f'away_{c}' for c in bp_cols})
games = games.merge(
    away_bp.assign(_t=away_bp['team']).drop(columns='team'),
    left_on=['game_date','away_team'], right_on=['game_date','_t'], how='left'
).drop(columns='_t')

games = games.drop_duplicates(subset=['game_date','home_team','away_team'], keep='first').reset_index(drop=True)

print(f"Games: {len(games)}")
print(f"Home bullpen matched: {games['home_bp_whiff_pct_talent'].notna().sum()}")
print(f"Away bullpen matched: {games['away_bp_whiff_pct_talent'].notna().sum()}")

# %%
# ============================================================
# 4. DROP unmatched + SAVE v3 feature set
#    (drop on the TALENT cols; rolling cols may be NaN earlier in season)
# ============================================================
before = len(games)
games_v3 = games.dropna(subset=[
    'home_bp_whiff_pct_talent', 'away_bp_whiff_pct_talent',
]).reset_index(drop=True)

print(f"Dropped {before - len(games_v3)} games missing bullpen data")
print(f"v3 dataset ({SEASON}): {len(games_v3)} games")

games_v3.to_csv(GAMES_V3_OUT, index=False)
print(f"Saved -> {GAMES_V3_OUT}")

This is a large query, it may take a moment to complete


/opt/anaconda3/lib/python3.13/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
  0%|          | 0/99 [00:00<?, ?it/s]/opt/anaconda3/lib/python3.13/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = 

  ...1/30 teams pulled
This is a large query, it may take a moment to complete


/opt/anaconda3/lib/python3.13/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
  0%|          | 0/99 [00:00<?, ?it/s]/opt/anaconda3/lib/python3.13/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = 

  ...2/30 teams pulled
This is a large query, it may take a moment to complete


...

  ...28/30 teams pulled
This is a large query, it may take a moment to complete


/opt/anaconda3/lib/python3.13/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
  1%|          | 1/99 [00:01<01:52,  1.14s/it]/opt/anaconda3/lib/python3.13/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[co

  ...29/30 teams pulled
This is a large query, it may take a moment to complete


/opt/anaconda3/lib/python3.13/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
  1%|          | 1/99 [00:01<01:49,  1.12s/it]/opt/anaconda3/lib/python3.13/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[co

  ...30/30 teams pulled

Done. 2493 team-game bullpen lines from 29 teams. Failed: ['CLE']
   team  game_date  bp_whiff_pct  bp_whiff_pct_roll10  bp_whiff_pct_talent
0   ATH 2026-03-27      0.153846                  NaN                  NaN
1   ATH 2026-03-28      0.280000                  NaN                  NaN
2   ATH 2026-03-29      0.150000                  NaN                  NaN
3   ATH 2026-03-30      0.344828             0.194615             0.194615
4   ATH 2026-03-31      0.166667             0.232168             0.232168
5   ATH 2026-04-01      0.205128             0.219068             0.219068
6   ATH 2026-04-03      0.333333             0.216745             0.216745
7   ATH 2026-04-04      0.236364             0.233400             0.233400
8   ATH 2026-04-05      0.200000             0.233771             0.233771
9   ATH 2026-04-07      0.187500             0.230018             0.230018
10  ATH 2026-04-08      0.555556             0.225767             0.225767
11  ATH 2